In [1]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecMonitor

### Add mbt-gym to path

In [2]:
import sys
sys.path.append("../")

In [3]:
from mbt_gym.agents.BaselineAgents import CarteaJaimungalMmAgent
from mbt_gym.gym.helpers.generate_trajectory import generate_trajectory
from mbt_gym.gym.StableBaselinesTradingEnvironment import StableBaselinesTradingEnvironment
from mbt_gym.gym.TradingEnvironment import TradingEnvironment
from mbt_gym.gym.wrappers import *
from mbt_gym.rewards.RewardFunctions import PnL, CjMmCriterion
from mbt_gym.stochastic_processes.midprice_models import BrownianMotionMidpriceModel
from mbt_gym.stochastic_processes.arrival_models import PoissonArrivalModel
from mbt_gym.stochastic_processes.fill_probability_models import ExponentialFillFunction
from mbt_gym.gym.ModelDynamics import LimitOrderModelDynamics

### Create market making environment

In [4]:
import sys
sys.path.append("../") # This version of the notebook is in the subfolder "notebooks" of the repo

import gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from copy import deepcopy


from mbt_gym.agents.BaselineAgents import *
from mbt_gym.gym.TradingEnvironment import TradingEnvironment
from mbt_gym.gym.helpers.generate_trajectory import generate_trajectory
from mbt_gym.gym.helpers.plotting import *
from mbt_gym.stochastic_processes.midprice_models import *
from mbt_gym.stochastic_processes.arrival_models import *
from mbt_gym.stochastic_processes.fill_probability_models import *
import torch
#print(torch.cuda.is_available())
#print(torch.cuda.get_device_name())
from mbt_gym.gym.ModelDynamics import LimitOrderModelDynamics
seed = 1


## Varying fad proportion (paramter q)

### Parameters

In [5]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 100
initial_inventory = 0
fill_exponent = 0
fads_proportions = [0.0, 0.2, 0.4, 0.6, 0.8, 1]
# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
baseline_arrival_rate = np.array([[5.0, 5.0]])
phi = 15
psi = 15
k = 1
gamma = 1
alpha=0.001
big_phi=0.1
mu=0

In [6]:
def get_as_env(num_trajectories:int = 1, fads_proportion:float=0.6):
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModelPartialInformation(initial_price=initial_price, drift=mu,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories)
    arrival_model = ModifiedPoissonArrivalModel(phi=phi,
                                                psi=psi,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma,
                                                step_size=terminal_time/n_steps,
                                                num_trajectories=num_trajectories,
                                                seed=seed)
    fill_probability_model = ExponentialFillFunction(fill_exponent=k, 
                                                     step_size=terminal_time/n_steps,
                                                     num_trajectories=num_trajectories,
                                                     seed=seed)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories)
    reward = CjMmCriterion(per_step_inventory_aversion = big_phi,
                           terminal_inventory_aversion = alpha,
                           terminal_time = terminal_time)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [7]:
results_dict = {}
for fads_proportion in fads_proportions:
    vec_env = get_as_env(num_trajectories=1000, fads_proportion=fads_proportion)

    vec_as = OptimizedPartialInfoMMwithFadsInformedUniformedTradersAgent(
      env=vec_env
    )

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[fads_proportion] = dict(results=results, rewards=total_rewards, obs=observations)

Starting precomputation with parameters:
phi=15, psi=15, eta=10.0
gamma=1, k=1, sigma=1.0
fads_proportion=0.0
✓ Pre-computed ODE solutions for 100 time points


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")
/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


Starting precomputation with parameters:
phi=15, psi=15, eta=10.0
gamma=1, k=1, sigma=1.0
fads_proportion=0.2
✓ Pre-computed ODE solutions for 100 time points
Starting precomputation with parameters:
phi=15, psi=15, eta=10.0
gamma=1, k=1, sigma=1.0
fads_proportion=0.4
✓ Pre-computed ODE solutions for 100 time points


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")
/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


Starting precomputation with parameters:
phi=15, psi=15, eta=10.0
gamma=1, k=1, sigma=1.0
fads_proportion=0.6
✓ Pre-computed ODE solutions for 100 time points
Starting precomputation with parameters:
phi=15, psi=15, eta=10.0
gamma=1, k=1, sigma=1.0
fads_proportion=0.8
✓ Pre-computed ODE solutions for 100 time points
Starting precomputation with parameters:
phi=15, psi=15, eta=10.0
gamma=1, k=1, sigma=1.0
fads_proportion=1
✓ Pre-computed ODE solutions for 100 time points


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")
/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


In [8]:
header = f"{'Fads Prop':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for fads_prop, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{fads_prop:10} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

 Fads Prop |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------
       0.0 |      -5.08 |      25.11 |          -0.015 | 19.218969145092043
       0.2 |      -5.17 |      25.27 |          -0.007 | 19.213249360792673
       0.4 |      -4.73 |      25.35 |           0.135 | 19.090122445914275
       0.6 |      -4.89 |      25.55 |          -0.039 | 19.140937255004
       0.8 |      -5.39 |      25.33 |          -0.253 | 19.275139195346945
         1 |      -8.34 |      26.34 |          -1.034 | 19.700072182608874


## Varying eta

### Parameters

In [12]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 100
initial_inventory = 0
fill_exponent = 0
fads_proportion = 0.6 # p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
etas = [2.5, 5, 7.5, 10.0, 12.5]
baseline_arrival_rate = np.array([[5.0, 5.0]])
phi = 15
psi = 15
k = 1
gamma = 1
alpha=0.001
big_phi=0.1
mu=0

In [13]:
def get_as_env(num_trajectories:int = 1, eta:float=10):
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModelPartialInformation(initial_price=initial_price, drift=mu,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories)
    arrival_model = ModifiedPoissonArrivalModel(phi=phi,
                                                psi=psi,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma,
                                                step_size=terminal_time/n_steps,
                                                num_trajectories=num_trajectories,
                                                seed=seed)
    fill_probability_model = ExponentialFillFunction(fill_exponent=k, 
                                                     step_size=terminal_time/n_steps,
                                                     num_trajectories=num_trajectories,
                                                     seed=seed)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories)
    reward = CjMmCriterion(per_step_inventory_aversion = big_phi,
                           terminal_inventory_aversion = alpha,
                           terminal_time = terminal_time)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [14]:
results_dict = {}
for eta in etas:
    vec_env = get_as_env(num_trajectories=1000, eta=eta)

    vec_as = OptimizedPartialInfoMMwithFadsInformedUniformedTradersAgent(
        env=vec_env
    )

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[eta] = dict(results=results, rewards=total_rewards, obs=observations)

Starting precomputation with parameters:
phi=15, psi=15, eta=2.5
gamma=1, k=1, sigma=1.0
fads_proportion=0.6
✓ Pre-computed ODE solutions for 100 time points
Starting precomputation with parameters:
phi=15, psi=15, eta=5
gamma=1, k=1, sigma=1.0
fads_proportion=0.6
✓ Pre-computed ODE solutions for 100 time points


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")
/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


Starting precomputation with parameters:
phi=15, psi=15, eta=7.5
gamma=1, k=1, sigma=1.0
fads_proportion=0.6
✓ Pre-computed ODE solutions for 100 time points
Starting precomputation with parameters:
phi=15, psi=15, eta=10.0
gamma=1, k=1, sigma=1.0
fads_proportion=0.6
✓ Pre-computed ODE solutions for 100 time points


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")
/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


Starting precomputation with parameters:
phi=15, psi=15, eta=12.5
gamma=1, k=1, sigma=1.0
fads_proportion=0.6
✓ Pre-computed ODE solutions for 100 time points


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


In [15]:
header = f"{'Eta':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for etas, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{etas:10} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

       Eta |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------
       2.5 |      -5.87 |      27.10 |          -0.025 | 19.42540540117503
         5 |      -5.05 |      26.06 |           0.117 | 19.162288772482267
       7.5 |      -4.91 |      25.79 |           0.062 | 19.150930943429355
      10.0 |      -4.89 |      25.55 |          -0.039 | 19.140937255004
      12.5 |      -4.82 |      25.50 |          -0.101 | 19.11525042995775


## Varying gamma parameter

### Parameters

In [19]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 100
initial_inventory = 0
fill_exponent = 0
fads_proportions = 0.6# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
phi = 15
psi = 15
k = 1
gammas = [0, 1, 2, 3]
alpha=0.001
big_phi=0.1
mu=0

In [30]:
def get_as_env(num_trajectories:int = 1, gamma:float=1):
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModel(initial_price=initial_price, drift=mu,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories)
    arrival_model = ModifiedPoissonArrivalModel(phi=phi,
                                                psi=psi,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma,
                                                step_size=terminal_time/n_steps,
                                                num_trajectories=num_trajectories,
                                                seed=seed)
    fill_probability_model = ExponentialFillFunction(fill_exponent=k, 
                                                     step_size=terminal_time/n_steps,
                                                     num_trajectories=num_trajectories,
                                                     seed=seed)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories)
    reward = CjMmCriterion(per_step_inventory_aversion = big_phi,
                           terminal_inventory_aversion = alpha,
                           terminal_time = terminal_time)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [33]:
results_dict = {}
for gamma in gammas:
    vec_env = get_as_env(num_trajectories=1000, gamma=gamma)

    vec_as = OptimizedPartialInfoMMwithFadsInformedUniformedTradersAgent(
        env=vec_env
    )

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[gamma] = dict(results=results, rewards=total_rewards, obs=observations)

Starting precomputation with parameters:
phi=30, psi=0, eta=10.0
gamma=0, k=1, sigma=1.0
fads_proportion=0.6
✓ Pre-computed ODE solutions for 100 time points
Starting precomputation with parameters:
phi=30, psi=0, eta=10.0
gamma=1, k=1, sigma=1.0
fads_proportion=0.6
✓ Pre-computed ODE solutions for 100 time points


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")
/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")
/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


Starting precomputation with parameters:
phi=30, psi=0, eta=10.0
gamma=2, k=1, sigma=1.0
fads_proportion=0.6
✓ Pre-computed ODE solutions for 100 time points
Starting precomputation with parameters:
phi=30, psi=0, eta=10.0
gamma=3, k=1, sigma=1.0
fads_proportion=0.6
✓ Pre-computed ODE solutions for 100 time points


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


In [34]:
header = f"{'Gamma':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for gamma, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{gamma:10} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

     Gamma |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------
         0 |    -469.40 |      92.66 |          29.621 | 4.637602721234323
         1 |    -469.40 |      92.66 |          29.621 | 4.637602721234323
         2 |    -469.40 |      92.66 |          29.621 | 4.637602721234323
         3 |    -469.40 |      92.66 |          29.621 | 4.637602721234323


## Varying Informed trader proportion (psi and phi)

### Parameters

In [35]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 100
initial_inventory = 0
fill_exponent = 0
fads_proportion = 0.6
# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
baseline_arrival_rate = np.array([[5.0, 5.0]])
phis = [30, 22.5, 15, 7.5, 0]
psis = [0, 7.5, 15, 22.5, 30]
k = 1
gamma = 1
alpha=0.001
big_phi=0.1
mu=0

In [36]:
def get_as_env(num_trajectories:int = 1, phi:float=15, psi:float=15):
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModel(initial_price=initial_price,drift=mu,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories)
    arrival_model = ModifiedPoissonArrivalModel(phi=phi,
                                                psi=psi,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma,
                                                step_size=terminal_time/n_steps,
                                                num_trajectories=num_trajectories,
                                                seed=seed)
    fill_probability_model = ExponentialFillFunction(fill_exponent=k, 
                                                     step_size=terminal_time/n_steps,
                                                     num_trajectories=num_trajectories,
                                                     seed=seed)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories)
    reward = CjMmCriterion(per_step_inventory_aversion = big_phi,
                           terminal_inventory_aversion = alpha,
                           terminal_time = terminal_time)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [37]:
results_dict = {}
for phi, psi in zip(phis, psis):
    vec_env = get_as_env(num_trajectories=1000, phi=phi, psi=psi)

    vec_as = OptimizedPartialInfoMMwithFadsInformedUniformedTradersAgent(
        env=vec_env
    )

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[(phi, psi)] = dict(results=results, rewards=total_rewards,  obs=observations)

Starting precomputation with parameters:
phi=30, psi=0, eta=10.0
gamma=1, k=1, sigma=1.0
fads_proportion=0.6
✓ Pre-computed ODE solutions for 100 time points


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")
/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


Starting precomputation with parameters:
phi=22.5, psi=7.5, eta=10.0
gamma=1, k=1, sigma=1.0
fads_proportion=0.6
✓ Pre-computed ODE solutions for 100 time points
Starting precomputation with parameters:
phi=15, psi=15, eta=10.0
gamma=1, k=1, sigma=1.0
fads_proportion=0.6
✓ Pre-computed ODE solutions for 100 time points
Starting precomputation with parameters:
phi=7.5, psi=22.5, eta=10.0
gamma=1, k=1, sigma=1.0
fads_proportion=0.6
✓ Pre-computed ODE solutions for 100 time points


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")
/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")
/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:1097: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


Starting precomputation with parameters:
phi=0, psi=30, eta=10.0
gamma=1, k=1, sigma=1.0
fads_proportion=0.6
✓ Pre-computed ODE solutions for 100 time points


In [38]:
header = f"{'Phi':>8} | {'Psi':>8} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for (phi, psi), result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{phi:8} | {psi:8} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

     Phi |      Psi |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
-------------------------------------------------------------------------------
      30 |        0 |    -469.40 |      92.66 |          29.621 | 4.637602721234323
    22.5 |      7.5 |    -505.15 |      95.84 |          29.644 | 4.661251334137647
      15 |       15 |    -536.39 |     102.54 |          29.733 | 4.720562572405963
     7.5 |     22.5 |    -562.30 |     111.17 |          29.733 | 4.775951318847377
       0 |       30 |    -587.87 |     123.30 |          29.808 | 4.891128295189158
